<a href="https://colab.research.google.com/github/Yash-k10/Machine_vision/blob/main/practical7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import cv2
import numpy as np
from google.colab import files

# ==========================
# Upload Video
# ==========================

uploaded = files.upload()

video_path = list(uploaded.keys())[0]

cap = cv2.VideoCapture(video_path)

# ==========================
# Video Information
# ==========================

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

if fps == 0:
    fps = 30

# ==========================
# Output Video
# ==========================

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    "vehicle_tracking_output.mp4",
    fourcc,
    fps,
    (width, height)
)

# ==========================
# Background Subtractor
# ==========================

bg_sub = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=False
)

# ==========================
# Read First Frame
# ==========================

ret, old_frame = cap.read()

if not ret:
    print("Error reading video.")
    exit()

old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

feature_params = dict(
    maxCorners=200,
    qualityLevel=0.3,
    minDistance=7,
    blockSize=7
)

lk_params = dict(
    winSize=(15,15),
    maxLevel=2,
    criteria=(cv2.TERM_CRITERIA_EPS |
              cv2.TERM_CRITERIA_COUNT,
              10,
              0.03)
)

p0 = cv2.goodFeaturesToTrack(old_gray,
                             mask=None,
                             **feature_params)

mask = np.zeros_like(old_frame)

frame_count = 0

# ==========================
# Processing Loop
# ==========================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # -----------------------
    # Background Subtraction
    # -----------------------

    fgmask = bg_sub.apply(frame)

    kernel = np.ones((5,5),np.uint8)

    fgmask = cv2.morphologyEx(
        fgmask,
        cv2.MORPH_OPEN,
        kernel
    )

    fgmask = cv2.dilate(fgmask,kernel,iterations=2)

    contours, _ = cv2.findContours(
        fgmask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # -----------------------
    # Motion Detection
    # -----------------------

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > 800:

            x,y,w,h = cv2.boundingRect(cnt)

            cv2.rectangle(
                frame,
                (x,y),
                (x+w,y+h),
                (0,255,0),
                2
            )

            cv2.putText(
                frame,
                "Vehicle",
                (x,y-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,255,0),
                2
            )

    # -----------------------
    # Optical Flow Tracking
    # -----------------------

    if p0 is not None:

        p1, st, err = cv2.calcOpticalFlowPyrLK(
            old_gray,
            gray,
            p0,
            None,
            **lk_params
        )

        if p1 is not None:

            good_new = p1[st==1]
            good_old = p0[st==1]

            for new, old in zip(good_new, good_old):

                a,b = new.ravel()
                c,d = old.ravel()

                a,b,c,d = int(a),int(b),int(c),int(d)

                cv2.line(
                    mask,
                    (a,b),
                    (c,d),
                    (0,0,255),
                    2
                )

                cv2.circle(
                    frame,
                    (a,b),
                    3,
                    (255,0,0),
                    -1
                )

            frame = cv2.add(frame, mask)

            old_gray = gray.copy()

            p0 = good_new.reshape(-1,1,2)

        else:

            p0 = cv2.goodFeaturesToTrack(
                gray,
                mask=None,
                **feature_params
            )

            old_gray = gray.copy()

    else:

        p0 = cv2.goodFeaturesToTrack(
            gray,
            mask=None,
            **feature_params
        )

        old_gray = gray.copy()

    # -----------------------
    # Write Output Video
    # -----------------------

    out.write(frame)

    frame_count += 1

    if frame_count % 50 == 0:
        print(f"Processed {frame_count} frames...")

# ==========================
# Release Resources
# ==========================

cap.release()
out.release()

print("Processing Completed!")
print("Output saved as vehicle_tracking_output.mp4")

# ==========================
# Download Output
# ==========================

files.download("vehicle_tracking_output.mp4")


Saving traffic.mp4 to traffic.mp4
Processed 50 frames...
Processed 100 frames...
Processed 150 frames...
Processed 200 frames...
Processed 250 frames...
Processed 300 frames...
Processed 350 frames...
Processed 400 frames...
Processed 450 frames...
Processed 500 frames...
Processed 550 frames...
Processed 600 frames...
Processed 650 frames...
Processed 700 frames...
Processed 750 frames...
Processed 800 frames...
Processed 850 frames...
Processed 900 frames...
Processed 950 frames...
Processed 1000 frames...
Processed 1050 frames...
Processed 1100 frames...
Processed 1150 frames...
Processed 1200 frames...
Processed 1250 frames...
Processed 1300 frames...
Processed 1350 frames...
Processed 1400 frames...
Processed 1450 frames...
Processed 1500 frames...
Processing Completed!
Output saved as vehicle_tracking_output.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>